## Modules

In [ ]:
import os 
import pickle

# pyrefly: ignore [missing-import]
import numpy as np
# pyrefly: ignore [missing-import]
from skimage.io import imread
from skimage.transform import resize

from sklearn.model_selection import train_test_split , GroupShuffleSplit

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
# pyrefly: ignore [missing-import]
from xgboost import XGBClassifier

from sklearn.metrics import classification_report , accuracy_score

## Data

In [ ]:
data = []
labels = []
groups = []

categories = ['empty', 'not_empty']
input = "data"

for cat_idx , cat in enumerate(categories):
    for file in os.listdir(os.path.join(input,cat)):
        img_path = os.path.join(input , cat , file)

        img = resize( imread(img_path) , (150,150))

        data.append(img.flatten())
        labels.append(cat_idx)
        groups.append(file.split('_')[1].split('.')[0])

data = np.asarray(data)
labels = np.asarray(labels)
groups = np.asarray(groups)

print(len(data),len(labels))

6090 6090


## Train / Test split

In [ ]:
"""
x_train , x_test , y_train , y_test = train_test_split( data , labels , 
                                                test_size = 0.2, 
                                                shuffle = True,
                                                stratify= labels,
                                                random_state=42)
"""

'\nx_train , x_test , y_train , y_test = train_test_split( data , labels , \n                                                test_size = 0.2, \n                                                shuffle = True,\n                                                stratify= labels,\n                                                random_state=42)\n'

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(data, labels, groups))

x_train = data[train_idx]
x_test = data[test_idx]
y_train = labels[train_idx]
y_test = labels[test_idx]

### SVMC

In [ ]:
svm = SVC()

param = [{'gamma' : [0.01 , 0.001, 0.0001],
        'C' : [1,10,100,1000]
        }]

grid_search = GridSearchCV( svm , param)

grid_search.fit(x_train , y_train)

KeyboardInterrupt: 

In [ ]:
svm = SVC(C=10, gamma=0.01)

svm.fit(x_train, y_train)

y_pred_svm = svm.predict(x_train)

print("--- SVM Results ---")
print(classification_report(y_train, y_pred_svm))

KeyboardInterrupt: 

### XGBoost

In [ ]:
xgb = XGBClassifier(
    n_estimators = 500,
    max_depth = 6,
    learning_rate = 0.05,

    # device = "cuda",
    tree_method = "hist",

    subsample = 0.8,
    colsample_bytree = 0.8,

    random_state = 42,
    eval_metric = 'logloss'
)

xgb.fit(x_train, y_train)

y_pred_xgb = xgb.predict(x_train)

print("--- XGBoost Results (Train) ---")
print(classification_report(y_train, y_pred_xgb))

--- XGBoost Results (Train) ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2371
           1       1.00      1.00      1.00      2469

    accuracy                           1.00      4840
   macro avg       1.00      1.00      1.00      4840
weighted avg       1.00      1.00      1.00      4840



### RandomForest

In [ ]:
rfc = RandomForestClassifier(n_estimators=100,max_depth=10,random_state=42,
                            oob_score=True, n_jobs=-1)

rfc.fit(x_train,y_train)

y_rf_pred = rfc.predict(x_train)

print("--- RANDOM FOREST Results (Train) ---")
print(classification_report(y_train, y_rf_pred))


--- RANDOM FOREST Results (Train) ---
              precision    recall  f1-score   support

           0       0.99      0.98      0.99      3346
           1       0.99      0.99      0.99      6721

    accuracy                           0.99     10067
   macro avg       0.99      0.99      0.99     10067
weighted avg       0.99      0.99      0.99     10067



KeyboardInterrupt: 

#### OOB

In [ ]:
oob_accuracy = rfc.oob_score_
print(f"OOB Score (Global Precision) : {oob_accuracy:.4f}")

OOB Score (Global Precision) : 1.0000


In [ ]:
y_pred_xgb = xgb.predict(x_test)
print("---- XGBoost ----")
print(classification_report(y_test,y_pred_xgb))
print("")

y_pred_rf = rfc.predict(x_test)
print("---- RandomForest ----")
print(classification_report(y_test,y_pred_rf))
print("")

score = accuracy_score ( svm.predict(x_test) , y_test )
print("---- SVM ---- : ",score)

---- XGBoost ----
              precision    recall  f1-score   support

           0       0.95      0.86      0.90       674
           1       0.85      0.95      0.90       576

    accuracy                           0.90      1250
   macro avg       0.90      0.90      0.90      1250
weighted avg       0.91      0.90      0.90      1250


---- RandomForest ----
              precision    recall  f1-score   support

           0       1.00      0.82      0.90       674
           1       0.83      1.00      0.90       576

    accuracy                           0.90      1250
   macro avg       0.91      0.91      0.90      1250
weighted avg       0.92      0.90      0.90      1250


---- SVM ---- :  0.9504


## loading the model

In [ ]:
pickle.dump(svm , open("./model.p" , "wb"))

In [ ]:
busy_A = "data_2/A/busy"
free_A = "data_2/A/free"

data_busy_A  , data_free_A = [] , []

for cat in [busy_A , free_A] : 
    if cat == busy_A :
        for file in os.listdir(busy_A):
            
            img_path = os.path.join(busy_A , file)

            img = resize( imread(img_path) , (100,100))
            data_busy_A.append(img.flatten())
    
    if cat == free_A :
        for file in os.listdir(free_A):
            
            img_path = os.path.join(free_A , file)

            img = resize( imread(img_path) , (100,100))
            data_free_A.append(img.flatten())

data_busy_A = np.asarray(data_busy_A)
data_free_A = np.asarray(data_free_A)

print(len(data_busy_A))
print(len(data_free_A))

3621
2550


In [ ]:
pred =svm.predict(data_free_A)
counts = np.bincount(pred)
print(f"Empty spaces (0): {counts[0]}")
print(f"Occupied spaces (1): {counts[1]}")

Empty spaces (0): 1902
Occupied spaces (1): 648


In [ ]:
pred = svm.predict(data_busy_A)

counts = np.bincount(pred)
print(f"Empty spaces (0): {counts[0]}")
print(f"Occupied spaces (1): {counts[1]}")


Empty spaces (0): 1399
Occupied spaces (1): 2222


In [ ]:
pred = xgb.predict(data_free_A)
counts = np.bincount(pred)
print(f"Empty spaces (0): {counts[0]}")
print(f"Occupied spaces (1): {counts[1]}")

Empty spaces (0): 722
Occupied spaces (1): 1828


In [ ]:
pred = xgb.predict(data_busy_A)
counts = np.bincount(pred)
print(f"Empty spaces (0): {counts[0]}")
print(f"Occupied spaces (1): {counts[1]}")

Empty spaces (0): 842
Occupied spaces (1): 2779


In [ ]:
pred = rfc.predict(data_free_A)
counts = np.bincount(pred)
print(f"Empty spaces (0): {counts[0]}")
print(f"Occupied spaces (1): {counts[1]}")

Empty spaces (0): 614
Occupied spaces (1): 1936


In [ ]:
pred = rfc.predict(data_busy_A)
counts = np.bincount(pred)
print(f"Empty spaces (0): {counts[0]}")
print(f"Occupied spaces (1): {counts[1]}")

Empty spaces (0): 520
Occupied spaces (1): 3101


# B

In [ ]:
busy_B = "data_2/B/busy"
free_B = "data_2/B/free"

data_busy_B , data_free_B= [] , []

for cat in [busy_B, free_B] : 
    if cat == busy_B:
        for file in os.listdir(busy_B):
            
            img_path = os.path.join(busy_B, file)

            img = resize( imread(img_path) , (15,15))
            data_busy_B.append(img.flatten())
    
    if cat == free_B:
        for file in os.listdir(free_B):
            
            img_path = os.path.join(free_B, file)

            img = resize( imread(img_path) , (15,15))
            data_free_B.append(img.flatten())

data_busy_B= np.asarray(data_busy_B)
data_free_B= np.asarray(data_free_B)

print(len(data_busy_B))
print(len(data_free_B))

4781
1632


In [ ]:
pred = rfc.predict(data_free_B)
counts = np.bincount(pred)
print(f"Empty spaces (0): {counts[0]}")
print(f"Occupied spaces (1): {counts[1]}")

Empty spaces (0): 531
Occupied spaces (1): 1101


In [ ]:
pred = rfc.predict(data_busy_B)
counts = np.bincount(pred)
print(f"Empty spaces (0): {counts[0]}")
print(f"Occupied spaces (1): {counts[1]}")

Empty spaces (0): 1308
Occupied spaces (1): 3473


In [ ]:
labels = [1] * len(data_busy_A) + [0] * len(data_free_A)

labels += [1] * len(data_busy_B) + [0] * len(data_free_B)

full_data = np.concatenate([data_busy_A, data_free_A, data_busy_B, data_free_B], axis=0)

In [ ]:
pred = rfc.predict(full_data)
print("---- RandomForest ----")
print(classification_report(labels,pred))
print("")

---- RandomForest ----
              precision    recall  f1-score   support

           0       0.99      0.98      0.98      4182
           1       0.99      0.99      0.99      8402

    accuracy                           0.99     12584
   macro avg       0.99      0.99      0.99     12584
weighted avg       0.99      0.99      0.99     12584




In [ ]:
pred = svm.predict(full_data)
print("---- SVM ----")
print(classification_report(labels,pred))
print("")

---- SVM ----
              precision    recall  f1-score   support

           0       0.47      0.75      0.58      4182
           1       0.82      0.59      0.69      8402

    accuracy                           0.64     12584
   macro avg       0.65      0.67      0.63     12584
weighted avg       0.71      0.64      0.65     12584




In [ ]:
pred = xgb.predict(full_data)
print("---- XGBoost ----")
print(classification_report(labels,pred))
print("")

---- XGBoost ----
              precision    recall  f1-score   support

           0       0.39      0.41      0.40      4182
           1       0.70      0.69      0.69      8402

    accuracy                           0.59     12584
   macro avg       0.55      0.55      0.55     12584
weighted avg       0.60      0.59      0.60     12584




In [ ]:
x_train , x_test , y_train , y_test = train_test_split( full_data , labels , 
                                                test_size = 0.2, 
                                                shuffle = True,
                                                stratify= labels,
                                                random_state=42)

In [ ]:
rfc = RandomForestClassifier(n_estimators=100,max_depth=10,random_state=42,
                            oob_score=True, n_jobs=-1)

rfc.fit(x_train,y_train)

y_rf_pred = rfc.predict(x_train)

print("--- RANDOM FOREST Results (Train) ---")
print(classification_report(y_train, y_rf_pred))


--- RANDOM FOREST Results (Train) ---
              precision    recall  f1-score   support

           0       0.99      0.98      0.99      3346
           1       0.99      0.99      0.99      6721

    accuracy                           0.99     10067
   macro avg       0.99      0.99      0.99     10067
weighted avg       0.99      0.99      0.99     10067



In [ ]:
pred = rfc.predict(x_test)
print("---- RandomForest ----")
print(classification_report(y_test,pred))
print("")

---- RandomForest ----
              precision    recall  f1-score   support

           0       0.97      0.97      0.97       836
           1       0.98      0.99      0.98      1681

    accuracy                           0.98      2517
   macro avg       0.98      0.98      0.98      2517
weighted avg       0.98      0.98      0.98      2517


